## PropertyLens RAG v3

**What changed vs v2:**
- `MODEL_BUNDLE_PATH` → `hybrid_cluster_bundle.joblib` (correct artefact)
- `load_model_bundle()` → calls `joblib.load()` directly, no dict check
- `run_prediction()` → calls `.predict()` on the bundle object directly
- `load_xai_bundle()` → reads `hybrid_xai/global_shap_cache.json`, `rules.json`, `cbr_training_data.parquet` directly
- `reciprocal_rank_fusion()` → source-aware weights: amenity=2.5, xai=2.5, trend=1.0, transaction=1.0
- System prompt → verdict only for price queries, not amenity/trend/xai queries
- Demo queries → 8 queries covering all personas + new PropertyGuru listing query
- `run_demo()` print → source-aware formatting per chunk type

### Prerequisites
- `.env` at repo root with `PINECONE_API_KEY=...`
- Ollama running: `ollama pull gemma3`
- `data/artifacts/hybrid_cluster_bundle.joblib` present
- `data/artifacts/hybrid_xai/` folder with `global_shap_cache.json`, `rules.json`, `cbr_training_data.parquet`


In [1]:
# pinecone + pinecone-text  → hybrid vector index + BM25 sparse encoder
# sentence-transformers     → BGE-M3 dense embeddings (local)
# transformers + torch      → cross-encoder reranker (BAAI/bge-reranker-v2-m3)
# ollama                    → Gemma 3 local LLM
# joblib                    → load HybridClusterEnsemble
# pyarrow                   → read cbr_training_data.parquet
# pandas / numpy / tqdm     → data processing
# python-dotenv             → .env secrets
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama joblib pyarrow pandas numpy tqdm python-dotenv

### Configuration

All secrets and tuneable knobs in one place.
- `MODEL_BUNDLE_PATH` now points at `hybrid_cluster_bundle.joblib`
- `SOURCE_WEIGHTS` controls RRF namespace boosting — amenity and xai are boosted to 2.5 so they surface in mixed-source queries


In [2]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone ───────────────────────────────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

# ── Models ─────────────────────────────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DIMENSION  = 1024
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Retrieval knobs ────────────────────────────────────────────────────────────
TOP_K_RETRIEVAL = 50
TOP_K_RERANK    = 10
TOP_K_MMR       = 5
TOP_K_FINAL     = 5
MMR_LAMBDA      = 0.7
RRF_K           = 60
N_SUBQUERIES    = 3

# ── RRF source weights (v3 fix) ────────────────────────────────────────────────
# Boost amenity and xai chunks so they surface against the large transactions pool
SOURCE_WEIGHTS: dict[str, float] = {
    "transaction": 1.0,
    "amenity":     2.5,
    "trend":       1.0,
    "xai":         2.5,
}

# When fitting BM25 from scratch (no cache), sample at most this many *transaction*
# chunk texts so laptops do not OOM. Set to None to use every transaction row.
# Smaller = less RAM at fit time; delete bm25_encoder_v3.pkl after changing this.
BM25_MAX_TRANSACTION_TEXTS: int | None = 400_000

# ── LLM ───────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"

# ── Repo / data paths ──────────────────────────────────────────────────────────
def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from cwd until we find a dir containing both data/ and notebooks/."""
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root — expected data/ and notebooks/ siblings.")

REPO_ROOT         = find_repo_root()
DATA_ROOT         = str(REPO_ROOT / "data")
TRANSACTIONS_GLOB = str(REPO_ROOT / "data" / "feature_data" / "**" / "outputs" / "*.csv")
AMENITIES_GLOB    = str(REPO_ROOT / "data" / "amenities" / "*.csv")
ARTIFACTS_DIR     = str(REPO_ROOT / "data" / "artifacts")
XAI_DIR           = str(REPO_ROOT / "data" / "artifacts" / "hybrid_xai")
MODEL_BUNDLE_PATH = str(REPO_ROOT / "data" / "artifacts" / "hybrid_cluster_bundle.joblib")

print("Config loaded.")
print(f"Pinecone index   : {PINECONE_INDEX}")
print(f"REPO_ROOT        : {REPO_ROOT}")
print(f"Model bundle     : {MODEL_BUNDLE_PATH}")
print(f"XAI dir          : {XAI_DIR}")
print(f"Source weights   : {SOURCE_WEIGHTS}")

Config loaded.
Pinecone index   : propertylens-rag
REPO_ROOT        : /Users/bhuvesh/Documents/PropertyLens
Model bundle     : /Users/bhuvesh/Documents/PropertyLens/data/artifacts/hybrid_cluster_bundle.joblib
XAI dir          : /Users/bhuvesh/Documents/PropertyLens/data/artifacts/hybrid_xai
Source weights   : {'transaction': 1.0, 'amenity': 2.5, 'trend': 1.0, 'xai': 2.5}


### Load raw data

Loads all four sources:
- `transactions_df` — HDB resale records
- `amenities_df` — MRT, schools, malls, hawker centres
- `trends_df` — derived median price per town per year
- `xai_bundle` — reads directly from `hybrid_xai/` folder (v3 fix)


In [3]:
from __future__ import annotations
import glob
import json
import joblib
import os
import pandas as pd


def _infer_from_onehots(df: pd.DataFrame, prefix: str) -> pd.Series | None:
    """Infer a categorical value from one-hot columns like town_* or flat_type_*."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return None
    return df[cols].idxmax(axis=1).str.replace(prefix, "", regex=False)


def load_transactions(pattern: str) -> pd.DataFrame:
    """Load all transaction CSVs matching glob pattern."""
    files = glob.glob(pattern, recursive=True)
    if not files:
        raise FileNotFoundError(f"No CSVs found: {pattern}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    if "town" not in df.columns:
        town = _infer_from_onehots(df, "town_")
        if town is not None:
            df["town"] = town

    if "flat_type" not in df.columns:
        flat_type = _infer_from_onehots(df, "flat_type_")
        if flat_type is not None:
            df["flat_type"] = flat_type

    if "town" in df.columns:
        df["town"] = df["town"].astype(str).str.upper().str.strip()
    if "flat_type" in df.columns:
        df["flat_type"] = df["flat_type"].astype(str).str.upper().str.strip()

    # derive transaction_year from 'month' column if needed
    if "transaction_year" not in df.columns and "month" in df.columns:
        df["transaction_year"] = pd.to_datetime(df["month"], errors="coerce").dt.year
    print(f"  Transactions : {len(df):,} rows from {len(files)} file(s)")
    return df


def load_amenities(pattern: str) -> pd.DataFrame:
    """Load all amenity CSVs, tagging each row with its source filename."""
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No CSVs found: {pattern}")
    dfs = []
    for f in files:
        tmp = pd.read_csv(f)
        tmp["source_file"] = os.path.basename(f)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Amenities    : {len(df):,} rows from {len(files)} file(s)")
    return df


def derive_trends(transactions: pd.DataFrame) -> pd.DataFrame:
    """Compute median resale_price per town per year."""
    if "town" not in transactions.columns:
        raise KeyError(
            "transactions_df has no 'town' column. "
            "Expected a 'town' field or one-hot columns named town_* (see load_transactions)."
        )

    year_col = "transaction_year" if "transaction_year" in transactions.columns else "year"
    if year_col not in transactions.columns and "month" in transactions.columns:
        transactions = transactions.copy()
        transactions["transaction_year"] = pd.to_datetime(
            transactions["month"], errors="coerce"
        ).dt.year
        year_col = "transaction_year"

    grp = (
        transactions.groupby(["town", year_col])["resale_price"]
        .agg(median_resale_price="median", transaction_count="count")
        .reset_index()
        .rename(columns={year_col: "year"})
    )
    print(f"  Trends       : {len(grp):,} town-year rows")
    return grp


def load_xai_bundle(xai_dir: str) -> dict:
    """
    Load XAI artefacts directly from data/artifacts/hybrid_xai/.

    Reads:
    - global_shap_cache.json  → bundle['global_shap']
    - rules.json              → bundle['rules']
    - cbr_training_data.parquet → bundle['cbr_data']

    Returns:
        dict with loaded artefacts (empty dict if xai_dir missing).
    """
    bundle: dict = {}
    if not os.path.isdir(xai_dir):
        print(f"  WARNING: XAI dir not found: {xai_dir}")
        return bundle

    shap_path = os.path.join(xai_dir, "global_shap_cache.json")
    if os.path.exists(shap_path):
        with open(shap_path) as f:
            bundle["global_shap"] = json.load(f)
        print(f"  XAI SHAP     : loaded global_shap_cache.json")

    rules_path = os.path.join(xai_dir, "rules.json")
    if os.path.exists(rules_path):
        with open(rules_path) as f:
            bundle["rules"] = json.load(f)
        print(f"  XAI rules    : loaded rules.json")

    cbr_path = os.path.join(xai_dir, "cbr_training_data.parquet")
    if os.path.exists(cbr_path):
        bundle["cbr_data"] = pd.read_parquet(cbr_path)
        print(f"  XAI CBR      : loaded cbr_training_data.parquet — {len(bundle['cbr_data']):,} rows")

    if not bundle:
        print(f"  WARNING: no XAI artefacts found in {xai_dir}")
    return bundle


print("Loading data sources...")
transactions_df = load_transactions(TRANSACTIONS_GLOB)
amenities_df    = load_amenities(AMENITIES_GLOB)
trends_df       = derive_trends(transactions_df)
xai_bundle      = load_xai_bundle(XAI_DIR)
print("Done.")

Loading data sources...
  Transactions : 1,568,804 rows from 9 file(s)
  Amenities    : 615 rows from 4 file(s)
  Trends       : 312 town-year rows
  XAI SHAP     : loaded global_shap_cache.json
  XAI rules    : loaded rules.json
  XAI CBR      : loaded cbr_training_data.parquet — 204,066 rows
Done.


### Build transaction chunks

Each transaction → child chunk (core facts) + parent chunk (child + town amenity summary).
Only the child text is embedded and BM25-fitted. Parent text is stored in Pinecone metadata
and sent to the LLM at generation time.


In [4]:
from __future__ import annotations
import math, re
from typing import Any
import numpy as np
import pandas as pd


def _safe_float(x: Any) -> float | None:
    """Safely cast to float; return None if NaN or unconvertible."""
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def _bucket_storey(storey_range: Any) -> str:
    """Convert storey range string like '07 TO 09' to a coarse band."""
    if storey_range is None or (isinstance(storey_range, float) and math.isnan(storey_range)):
        return "unknown"
    m = re.search(r"(\d{1,2})\s*TO\s*(\d{1,2})", str(storey_range).upper())
    mid = (int(m.group(1)) + int(m.group(2))) / 2 if m else None
    if mid is None:
        m2 = re.search(r"(\d{1,2})", str(storey_range))
        mid = float(m2.group(1)) if m2 else None
    if mid is None: return "unknown"
    if mid <= 5:  return "01-05"
    if mid <= 12: return "06-12"
    if mid <= 20: return "13-20"
    return "21+"


def _bucket_price(price: float) -> str:
    """Bucket price into 100k bands, e.g. '500k-600k'."""
    if not np.isfinite(price) or price <= 0:
        return "unknown"
    lo = int(price // 100_000) * 100
    return f"{lo}k-{lo + 100}k"


HDB_TOWNS = (
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT BATOK", "BUKIT MERAH",
    "BUKIT PANJANG", "BUKIT TIMAH", "CENTRAL AREA", "CHOA CHU KANG",
    "CLEMENTI", "GEYLANG", "HOUGANG", "JURONG EAST", "JURONG WEST",
    "KALLANG/WHAMPOA", "MARINE PARADE", "PASIR RIS", "PUNGGOL",
    "QUEENSTOWN", "SEMBAWANG", "SENGKANG", "SERANGOON", "TAMPINES",
    "TOA PAYOH", "WOODLANDS", "YISHUN",
)


def _infer_town(text: str) -> str:
    """Best-effort town extraction from free-text address."""
    u = str(text).upper()
    for t in sorted(HDB_TOWNS, key=len, reverse=True):
        if t in u:
            return t
    return ""


def _ensure_town_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add town column if missing, inferred from address field."""
    out = df.copy()
    if "town" in out.columns and out["town"].notna().any():
        out["town"] = out["town"].str.upper().str.strip()
        return out
    if "address" in out.columns:
        out["town"] = out["address"].map(_infer_town)
    else:
        out["town"] = ""
    return out


def build_town_amenity_summary(amenities: pd.DataFrame) -> dict[str, str]:
    """
    Build a town → compact amenity summary string mapping.

    Returns:
        dict[UPPERCASE_TOWN, multi-line summary]
    """
    amenities = _ensure_town_column(amenities)
    summary: dict[str, str] = {}
    for town, grp in amenities.groupby("town"):
        if not str(town).strip():
            continue
        lines: list[str] = []
        if "source_file" in grp.columns:
            for src, sg in grp.groupby("source_file"):
                atype  = str(src).replace(".csv", "").replace("_", " ").title()
                names  = sg["name"].dropna().astype(str).tolist()
                ex     = ", ".join(names[:3])
                suffix = ", ..." if len(names) > 3 else ""
                lines.append(f"{atype} ({len(names)}): {ex}{suffix}")
        else:
            names = grp["name"].dropna().astype(str).tolist()
            lines.append(f"Amenities ({len(names)}): {', '.join(names[:5])}")
        summary[str(town).upper().strip()] = "\n".join(lines)
    return summary


def _format_child(row: pd.Series) -> str:
    """Short child chunk (~256 tokens) — core transaction facts."""
    town      = str(row.get("town", "")).upper().strip()
    flat_type = str(row.get("flat_type", "")).upper()
    year      = int(row.get("transaction_year", 0) or 0)
    price     = _safe_float(row.get("resale_price")) or 0.0
    area      = _safe_float(row.get("floor_area_sqm"))
    psf       = (price / area / 10.7639) if area and area > 0 else None
    storey    = _bucket_storey(row.get("storey_range"))
    ctx       = f"This is an HDB resale transaction in {town}, year {year}."
    facts = [
        f"Town: {town}",
        f"Flat type: {flat_type}",
        f"Storey band: {storey}",
        f"Floor area (sqm): {area:.1f}" if area else "Floor area (sqm): unknown",
        f"Resale price (SGD): {int(round(price))}",
        f"Approx PSF (SGD): {psf:.1f}" if psf else "Approx PSF (SGD): unknown",
    ]
    return ctx + "\n" + "\n".join(facts)


def _format_parent(row: pd.Series, amenity_summary: str) -> str:
    """Rich parent chunk (~1024 tokens) — child + town amenity summary."""
    base  = _format_child(row)
    lines = [base, "", "Nearby amenities (town-level):"]
    lines.append(amenity_summary if amenity_summary else "- No amenity data for this town.")
    return "\n".join(lines)


def build_transaction_chunks(
    transactions: pd.DataFrame,
    town_amenity_summary: dict[str, str],
) -> list[dict]:
    """
    Build Pinecone-ready chunk dicts for all HDB transactions.

    Args:
        transactions: HDB transactions DataFrame.
        town_amenity_summary: from build_town_amenity_summary().

    Returns:
        List of chunk dicts.
    """
    out: list[dict] = []
    for i, row in transactions.reset_index(drop=True).iterrows():
        town      = str(row.get("town", "")).upper().strip()
        flat_type = str(row.get("flat_type", "")).upper()
        year      = int(row.get("transaction_year", 0) or 0)
        price     = _safe_float(row.get("resale_price")) or 0.0
        area      = _safe_float(row.get("floor_area_sqm"))
        psf       = (price / area / 10.7639) if area and area > 0 else None
        addr      = str(row.get("address_key", f"row_{i}")).upper().replace(" ", "_")
        out.append({
            "id":          f"txn_{addr}_{year}",
            "text":        _format_child(row),
            "parent_text": _format_parent(row, town_amenity_summary.get(town, "")),
            "metadata": {
                "source":       "transaction",
                "town":         town,
                "flat_type":    flat_type,
                "storey_band":  _bucket_storey(row.get("storey_range")),
                "sale_year":    year,
                "price_band":   _bucket_price(price),
                "resale_price": int(round(price)),
                "psf":          float(round(psf, 1)) if psf is not None else None,
            },
        })
    return out


town_amenity_summary = build_town_amenity_summary(amenities_df)
txn_chunks = build_transaction_chunks(transactions_df, town_amenity_summary)
print(f"Transaction chunks : {len(txn_chunks):,}")
print("\nExample child:\n",  txn_chunks[0]["text"])
print("\nExample parent (400 chars):\n", txn_chunks[0]["parent_text"][:400])

Transaction chunks : 1,568,804

Example child:
 This is an HDB resale transaction in ANG MO KIO, year 2023.
Town: ANG MO KIO
Flat type: 2 ROOM
Storey band: unknown
Floor area (sqm): 44.0
Resale price (SGD): 267000
Approx PSF (SGD): 563.8

Example parent (400 chars):
 This is an HDB resale transaction in ANG MO KIO, year 2023.
Town: ANG MO KIO
Flat type: 2 ROOM
Storey band: unknown
Floor area (sqm): 44.0
Resale price (SGD): 267000
Approx PSF (SGD): 563.8

Nearby amenities (town-level):
Hawker Centres (7): MARKET & HAWKER CENTRE (BLK 409 ANG MO KIO AVE 10), MARKET & HAWKER CENTRE (BLK 724 ANG MO KIO AVE 6), CHENG SAN MARKET AND COOKED FOOD CENTRE, ...
Malls (2):


### Build standalone amenity chunks

One chunk per (town, amenity type) pair, stored in the `amenities` namespace.
Allows direct retrieval for queries like *"What MRT stations are near Bedok?"*
without relying on a transaction chunk to carry the info.


In [5]:
from __future__ import annotations
import pandas as pd


def build_amenity_chunks(amenities: pd.DataFrame) -> list[dict]:
    """
    Build one chunk per (town, amenity_type) pair.

    Args:
        amenities: amenities DataFrame.

    Returns:
        List of chunk dicts.
    """
    amenities   = _ensure_town_column(amenities)
    group_cols  = ["town", "source_file"] if "source_file" in amenities.columns else ["town"]
    out: list[dict] = []

    for keys, grp in amenities.groupby(group_cols):
        town  = str(keys[0] if isinstance(keys, tuple) else keys).upper().strip()
        if not town:
            continue
        src   = str(keys[1] if isinstance(keys, tuple) and len(keys) > 1 else "amenities").replace(".csv", "")
        atype = src.replace("_", " ").title()
        names = grp["name"].dropna().astype(str).tolist()
        text  = (
            f"{atype} in {town} ({len(names)} total): {', '.join(names)}. "
            f"These are the {atype.lower()} amenities in the {town} HDB town."
        )
        out.append({
            "id":          f"amenity_{town}_{src}".replace(" ", "_").lower(),
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":       "amenity",
                "town":         town,
                "amenity_type": atype,
                "count":        len(names),
            },
        })
    return out


amenity_chunks = build_amenity_chunks(amenities_df)
print(f"Amenity chunks : {len(amenity_chunks):,}")
print("\nExample:\n", amenity_chunks[0]["text"][:300])

Amenity chunks : 85

Example:
 Hawker Centres in ANG MO KIO (7 total): MARKET & HAWKER CENTRE (BLK 409 ANG MO KIO AVE 10), MARKET & HAWKER CENTRE (BLK 724 ANG MO KIO AVE 6), CHENG SAN MARKET AND COOKED FOOD CENTRE, CHONG BOON MARKET AND FOOD CENTRE, MAYFLOWER MARKET AND FOOD CENTRE, KEBUN BARU FOOD CENTRE, KEBUN BARU MARKET AND F


### Build trend chunks

One chunk per town-year. Stored in the `trends` namespace.
Answers questions like *"Are Tampines prices rising?"* without scanning transactions.


In [6]:
from __future__ import annotations
import pandas as pd


def build_trend_chunks(trends: pd.DataFrame) -> list[dict]:
    """
    One chunk per town-year from the trends DataFrame.

    Args:
        trends: output of derive_trends().

    Returns:
        List of chunk dicts.
    """
    out: list[dict] = []
    for _, row in trends.iterrows():
        town = str(row.get("town", "")).upper().strip()
        year = int(row.get("year", 0) or 0)
        med  = float(row.get("median_resale_price", 0) or 0)
        n    = int(row.get("transaction_count", 0) or 0)
        text = (
            f"HDB resale trend for {town}, year {year}: "
            f"median resale price SGD {int(round(med)):,} across {n:,} transactions."
        )
        out.append({
            "id":          f"trend_{town}_{year}",
            "text":        text,
            "parent_text": text,
            "metadata": {
                "source":            "trend",
                "town":              town,
                "sale_year":         year,
                "resale_price":      int(round(med)),
                "transaction_count": n,
            },
        })
    return out


trend_chunks = build_trend_chunks(trends_df)
print(f"Trend chunks : {len(trend_chunks):,}")
print("\nExample:\n", trend_chunks[0]["text"])

Trend chunks : 312

Example:
 HDB resale trend for ANG MO KIO, year 2015: median resale price SGD 355,000 across 5,390 transactions.


### Build XAI chunks

Reads directly from `data/artifacts/hybrid_xai/` (v3 fix — no longer looks inside model bundle):
- `global_shap_cache.json` → SHAP global feature importances
- `rules.json` → surrogate / apriori rules
- `cbr_training_data.parquet` → CBR comparable cases

Stored in the `xai` namespace. RRF weight = 2.5 so these surface for explanation queries.


In [7]:
from __future__ import annotations
import pandas as pd


def _shap_chunks(bundle: dict) -> list[dict]:
    """Build SHAP global importance chunks from bundle['global_shap']."""
    out: list[dict] = []
    shap_data = bundle.get("global_shap")
    if shap_data is None:
        print("  SHAP: key 'global_shap' not found — skipping.")
        return out
    if isinstance(shap_data, dict):
        items = sorted(shap_data.items(), key=lambda x: abs(float(x[1])), reverse=True)[:20]
    elif isinstance(shap_data, list):
        items = shap_data[:20]
    else:
        print("  SHAP: unexpected format — skipping.")
        return out
    lines = [f"  {feat}: {val:.4f}" for feat, val in items]
    text  = "Global SHAP feature importances for HDB price prediction:\n" + "\n".join(lines)
    out.append({
        "id": "xai_shap_global",
        "text": text,
        "parent_text": text,
        "metadata": {"source": "xai", "xai_type": "shap_global"},
    })
    return out


def _rule_chunks(bundle: dict) -> list[dict]:
    """Build rule chunks from bundle['rules']."""
    out: list[dict] = []
    rules = bundle.get("rules")
    if not rules:
        print("  Rules: key 'rules' not found — skipping.")
        return out
    rule_list = rules if isinstance(rules, list) else list(rules.items())[:50]
    for i, rule in enumerate(rule_list[:50]):
        text = f"HDB pricing rule #{i+1}: {str(rule)}"
        out.append({
            "id": f"xai_rule_{i}",
            "text": text,
            "parent_text": text,
            "metadata": {"source": "xai", "xai_type": "rule", "rule_id": i},
        })
    return out


def _cbr_chunks(bundle: dict) -> list[dict]:
    """Build CBR comparable case chunks from bundle['cbr_data']."""
    out: list[dict] = []
    cbr = bundle.get("cbr_data")
    if cbr is None:
        print("  CBR: key 'cbr_data' not found — skipping.")
        return out
    rows = cbr.head(200).to_dict(orient="records") if isinstance(cbr, pd.DataFrame) else cbr[:200]
    for i, case in enumerate(rows):
        text = "CBR comparable case: " + ", ".join(f"{k}={v}" for k, v in list(case.items())[:12])
        out.append({
            "id": f"xai_cbr_{i}",
            "text": text,
            "parent_text": text,
            "metadata": {
                "source":   "xai",
                "xai_type": "cbr",
                "town":     str(case.get("town", "")).upper(),
            },
        })
    return out


def build_xai_chunks(bundle: dict) -> list[dict]:
    """
    Build all XAI chunks from the loaded bundle dict.

    Args:
        bundle: output of load_xai_bundle().

    Returns:
        Combined list of SHAP + rule + CBR chunk dicts.
    """
    shap  = _shap_chunks(bundle)
    rules = _rule_chunks(bundle)
    cbr   = _cbr_chunks(bundle)
    print(f"  SHAP: {len(shap)} | Rules: {len(rules)} | CBR: {len(cbr)}")
    return shap + rules + cbr


xai_chunks = build_xai_chunks(xai_bundle)
print(f"XAI chunks total : {len(xai_chunks):,}")

  SHAP: 1 | Rules: 3 | CBR: 200
XAI chunks total : 204


### Initialise embedding models

- BGE-M3 dense encoder (1024-dim) for semantic similarity
- BM25Encoder fitted on chunk texts (cached under `notebooks/05_chatbot/bm25_encoder_v3.pkl`). If the cache exists, **no** 1.57M-row text list is built. Without cache, `BM25_MAX_TRANSACTION_TEXTS` caps how many transaction rows participate in `fit()` to avoid RAM spikes.


In [8]:
from __future__ import annotations
import pickle
import random
from pathlib import Path
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def load_dense_encoder(model_name: str) -> SentenceTransformer:
    """Load the BGE-M3 dense encoder."""
    return SentenceTransformer(model_name)


BM25_CACHE_PATH = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"


def _bm25_corpus_texts(
    txn_chunks: list,
    amenity_chunks: list,
    trend_chunks: list,
    xai_chunks: list,
    max_txn: int | None,
    seed: int = 42,
) -> list[str]:
    """Build text list for BM25.fit; optionally subsample transactions to save RAM."""
    rng = random.Random(seed)
    txn = txn_chunks
    if max_txn is not None and len(txn_chunks) > max_txn:
        txn = rng.sample(txn_chunks, max_txn)
        print(
            f"  BM25 fit corpus: {len(txn):,} / {len(txn_chunks):,} "
            f"transaction texts (BM25_MAX_TRANSACTION_TEXTS={max_txn})"
        )
    merged = txn + amenity_chunks + trend_chunks + xai_chunks
    return [c["text"] for c in merged]


def _fit_and_cache_bm25(corpus_texts: list[str], cache_path: Path) -> BM25Encoder:
    print(f"  BM25: fitting on {len(corpus_texts):,} documents...")
    enc = BM25Encoder()
    enc.fit(corpus_texts)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as f:
        pickle.dump(enc, f)
    print(f"  BM25: fitted and cached → {cache_path}")
    return enc


dense_encoder = load_dense_encoder(DENSE_MODEL_NAME)

if BM25_CACHE_PATH.exists():
    print(f"  BM25: loading from cache {BM25_CACHE_PATH}")
    with BM25_CACHE_PATH.open("rb") as f:
        bm25_encoder = pickle.load(f)
else:
    corpus_texts = _bm25_corpus_texts(
        txn_chunks,
        amenity_chunks,
        trend_chunks,
        xai_chunks,
        BM25_MAX_TRANSACTION_TEXTS,
    )
    bm25_encoder = _fit_and_cache_bm25(corpus_texts, BM25_CACHE_PATH)
    del corpus_texts

print(f"Dense encoder : {DENSE_MODEL_NAME}")
print("BM25 encoder  : ready")

/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 42797.90it/s]


  BM25: loading from cache /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
Dense encoder : BAAI/bge-m3
BM25 encoder  : ready


### Initialise Pinecone index

Single hybrid index (`dotproduct` metric), four namespaces.


In [9]:
from __future__ import annotations
from pinecone import Pinecone, ServerlessSpec


def init_pinecone(api_key: str) -> Pinecone:
    """Initialise and return a Pinecone client."""
    return Pinecone(api_key=api_key)


def get_or_create_index(pc: Pinecone, index_name: str, dimension: int) -> object:
    """
    Get existing or create a new Pinecone hybrid index.

    Uses dotproduct metric (required for hybrid search).
    """
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        print(f"  Creating index '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
    else:
        print(f"  Index '{index_name}' already exists.")
    return pc.Index(index_name)


pc    = init_pinecone(PINECONE_API_KEY)
index = get_or_create_index(pc, PINECONE_INDEX, DENSE_DIMENSION)
print(index.describe_index_stats())

  Index 'propertylens-rag' already exists.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '155',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:44:32 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '1',
                                    'x-pinecone-request-latency-ms': '0',
                                    'x-pinecone-response-duration-ms': '2'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


### Encode and upsert all chunks

Each chunk encoded with BGE-M3 (dense) + BM25 (sparse), upserted into its namespace.
`parent_text` stored in Pinecone metadata for LLM retrieval.

Set `SAMPLE_FOR_TESTING = False` to ingest the full 1.57M transactions.


In [10]:
from __future__ import annotations
import numpy as np
from tqdm import tqdm

SAMPLE_FOR_TESTING = True
SAMPLE_SIZE        = 1000


def _encode_one(
    chunk: dict,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
) -> dict:
    """Encode one chunk into a Pinecone upsert record."""
    dense  = dense_encoder.encode(chunk["text"], normalize_embeddings=True).tolist()
    sparse = bm25_encoder.encode_documents([chunk["text"]])[0]
    meta   = dict(chunk["metadata"])
    meta["parent_text"] = chunk["parent_text"][:3000]
    return {"id": chunk["id"], "values": dense, "sparse_values": sparse, "metadata": meta}


def upsert_namespace(
    index,
    chunks: list[dict],
    namespace: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    batch_size: int = 100,
) -> None:
    """
    Encode and upsert chunks into the given Pinecone namespace in batches.

    Encodes one batch at a time so full corpus upserts do not materialize
    all vectors in RAM at once.
    """
    n = len(chunks)
    for i in tqdm(range(0, n, batch_size), desc=f"Encoding {namespace}", total=(n + batch_size - 1) // batch_size):
        batch = chunks[i : i + batch_size]
        records = [_encode_one(c, dense_encoder, bm25_encoder) for c in batch]
        index.upsert(vectors=records, namespace=namespace)
    print(f"  Upserted {n:,} → '{namespace}'")


if SAMPLE_FOR_TESTING:
    import random
    sample = random.sample(txn_chunks, min(SAMPLE_SIZE, len(txn_chunks)))
    print(f"SAMPLE MODE: {len(sample):,} transaction chunks")
else:
    sample = txn_chunks
    print(f"FULL MODE: {len(sample):,} transaction chunks")

upsert_namespace(index, sample,         NS_TRANSACTIONS, dense_encoder, bm25_encoder)
upsert_namespace(index, amenity_chunks, NS_AMENITIES,    dense_encoder, bm25_encoder)
upsert_namespace(index, trend_chunks,   NS_TRENDS,       dense_encoder, bm25_encoder)
upsert_namespace(index, xai_chunks,     NS_XAI,          dense_encoder, bm25_encoder)

print("\nAll namespaces upserted.")
print(index.describe_index_stats())

SAMPLE MODE: 1,000 transaction chunks


Encoding transactions: 100%|██████████| 10/10 [00:37<00:00,  3.77s/it]


  Upserted 1,000 → 'transactions'


Encoding amenities: 100%|██████████| 1/1 [00:06<00:00,  6.11s/it]


  Upserted 85 → 'amenities'


Encoding trends: 100%|██████████| 4/4 [00:09<00:00,  2.48s/it]


  Upserted 312 → 'trends'


Encoding xai: 100%|██████████| 3/3 [00:09<00:00,  3.22s/it]


  Upserted 204 → 'xai'

All namespaces upserted.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 04:45:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 1000},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 204}},
 'storageFullness': 0.0,
 'total_vector_coun

### NLP filter extractor + namespace router

Extracts `town`, `flat_type`, `sale_year` from free-text via Gemma 3.
Namespace router selects which Pinecone namespaces to query based on keywords.


In [11]:
from __future__ import annotations
import json, re
import ollama


def _set_ollama_host(base_url: str) -> None:
    """Configure Ollama client host."""
    os.environ["OLLAMA_HOST"] = base_url


def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """
    Use Gemma 3 to extract Pinecone metadata filters from a free-text query.

    Returns dict with town/flat_type/sale_year or None if uncertain.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": ALL CAPS HDB town e.g. "TAMPINES", "BEDOK", "SERANGOON"
  - "flat_type": one of "2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"
  - "sale_year": integer year if mentioned
Omit any field you are not sure about. Return {{}} if nothing is clear.
Return ONLY JSON, no explanation.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if parsed else None
    except Exception as e:
        print(f"  Filter extraction failed: {e}")
        return None


def route_namespaces(query: str) -> list[str]:
    """
    Select Pinecone namespaces to query based on keywords.

    Always includes transactions. Adds:
    - amenities  : MRT, school, mall, hawker, near, amenity
    - trends     : trend, rising, falling, increase, history, recent
    - xai        : explain, shap, feature, why, reason, driver, factor
    """
    q  = query.lower()
    ns = [NS_TRANSACTIONS]
    if any(kw in q for kw in ["mrt", "school", "mall", "hawker", "near", "amenity", "transport", "bus"]):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in ["trend", "rising", "falling", "increase", "decrease", "history",
                               "recent", "last year", "past", "over time", "appreciation"]):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in ["explain", "shap", "feature", "why", "reason", "driver",
                               "factor", "importan", "predict", "model say"]):
        ns.append(NS_XAI)
    return ns


# ── smoke test ─────────────────────────────────────────────────────────────────
test_q = "Is $580k fair for a 4-room in Tampines?"
print(f"Query      : {test_q}")
print(f"Filters    : {extract_filters_from_query(test_q)}")
print(f"Namespaces : {route_namespaces(test_q)}")

Query      : Is $580k fair for a 4-room in Tampines?
Filters    : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
Namespaces : ['transactions']


### Hybrid retrieval + RRF with source weights (v3 fix)

Key change: `reciprocal_rank_fusion()` now reads `SOURCE_WEIGHTS` and multiplies
the RRF score contribution of each result by its source weight.
This ensures amenity (2.5×) and xai (2.5×) chunks surface in mixed-source queries
instead of being buried by the much larger transactions pool.


In [12]:
from __future__ import annotations
from typing import Any, Optional
import numpy as np


def _scale_sparse(sparse: dict, scale: float) -> dict:
    """Scale sparse vector values for alpha blending."""
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    alpha: float,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict[str, Any]]:
    """Single Pinecone hybrid query. alpha=1.0 → pure dense, 0.0 → pure sparse."""
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))
    res    = index.query(
        vector=dense, sparse_vector=sparse, top_k=int(top_k),
        namespace=namespace, include_metadata=True, filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id": getattr(m, "id", m.get("id")),
         "score": getattr(m, "score", m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


def retrieve_from_namespace(
    index,
    query: str,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    namespace: str,
    top_k: int,
    metadata_filter: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """
    Run dense + sparse retrieval from one namespace.

    Returns:
        (dense_results, sparse_results) for RRF fusion.
    """
    dense  = _hybrid_query(index, query, dense_encoder, bm25_encoder,
                           alpha=1.0, top_k=top_k, namespace=namespace,
                           metadata_filter=metadata_filter)
    sparse = _hybrid_query(index, query, dense_encoder, bm25_encoder,
                           alpha=0.0, top_k=top_k, namespace=namespace,
                           metadata_filter=metadata_filter)
    return dense, sparse


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
    source_weights: dict[str, float] | None = None,
) -> list[dict[str, Any]]:
    """
    Merge ranked lists using Reciprocal Rank Fusion with source-aware weights.

    score(d) = Σ  weight(source) × 1/(k + rank_i(d))

    Args:
        ranked_lists: each list sorted by relevance descending.
        k: RRF constant (default 60).
        source_weights: dict mapping source name to multiplier.
                        Defaults to SOURCE_WEIGHTS from config.

    Returns:
        Deduplicated list sorted by rrf_score descending.
    """
    weights = source_weights or SOURCE_WEIGHTS
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}

    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid    = str(r.get("id", ""))
            if not rid:
                continue
            source = str((r.get("metadata") or {}).get("source", "transaction"))
            weight = weights.get(source, 1.0)
            scores[rid] = scores.get(rid, 0.0) + weight * (1.0 / (float(k) + float(rank)))
            if rid not in best:
                best[rid] = r

    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


print("Retrieval functions defined (RRF source weights active).")

Retrieval functions defined (RRF source weights active).


### Reranking funnel — cross-encoder → MMR → lost-in-middle reorder


In [13]:
from __future__ import annotations
from typing import Any, Tuple
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_cross_encoder(model_name: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model. Returns (tokenizer, model)."""
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    return tok, model


def _get_text(candidate: dict, field: str = "parent_text") -> str:
    """Extract display text from a candidate's metadata."""
    md = candidate.get("metadata") or {}
    return str(md.get(field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
) -> list[dict[str, Any]]:
    """
    Score (query, passage) pairs with the cross-encoder; return top_k.

    Args:
        candidates: RRF-fused results.
        top_k: number to keep.

    Returns:
        Top-k candidates with ce_score added, sorted descending.
    """
    if not candidates:
        return []
    pairs  = [(query, _get_text(c)) for c in candidates]
    inputs = tokenizer(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    """Cosine similarity between two numpy vectors."""
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 1e-12 else 0.0


def mmr_filter(
    candidates: list[dict[str, Any]],
    dense_encoder: SentenceTransformer,
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
) -> list[dict[str, Any]]:
    """
    Select top_k diverse candidates via Maximal Marginal Relevance.

    Maximises: λ·relevance(d,q) − (1−λ)·max_cosine(d, selected)
    """
    if not candidates:
        return []
    texts   = [_get_text(c) for c in candidates]
    doc_emb = np.asarray(
        dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32
    )
    selected:  list[int] = []
    remaining: list[int] = list(range(len(candidates)))
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)
    while remaining and len(selected) < int(top_k):
        best_idx, best_val = None, -1e18
        for i in remaining:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score   = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val, best_idx = score, i
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [candidates[i] for i in selected]


def reorder_for_context_window(
    candidates: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Mitigate lost-in-the-middle: place best chunk first, second-best last.
    """
    if len(candidates) <= 2:
        return list(candidates)
    ordered = sorted(
        candidates,
        key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)),
        reverse=True,
    )
    return [ordered[0], *ordered[2:], ordered[1]]


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL)
print("Cross-encoder loaded.")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 19495.01it/s]

Cross-encoder loaded.


### Multi-query retrieval

Gemma 3 generates 3 reformulations of the original query.
All 4 queries (original + 3) run independently per namespace; all lists fuse via RRF.


In [14]:
from __future__ import annotations
import re
import ollama


def generate_subqueries(
    query: str,
    n: int = N_SUBQUERIES,
    model: str = OLLAMA_MODEL,
) -> list[str]:
    """
    Generate n reformulations of the query using Gemma 3.

    Returns:
        List of sub-query strings (may be shorter than n if generation fails).
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""You are a search query generator for Singapore HDB property data.
Generate {n} alternative search queries to help retrieve relevant data from a vector
database of HDB transactions, amenities, price trends, and SHAP features.
Return ONLY a numbered list. No explanations.

Original: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "")
        lines    = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception as e:
        print(f"  Sub-query generation failed: {e}")
        return []


def multi_query_retrieve(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    namespaces: list[str],
    top_k: int,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """
    Multi-query hybrid retrieval across all routed namespaces.

    Steps:
    1. Generate N sub-queries.
    2. For each (query + sub-query) × namespace: run dense + sparse.
    3. RRF-fuse all ranked lists with source weights.

    Args:
        metadata_filter: applied only to transactions namespace.

    Returns:
        RRF-fused candidates.
    """
    all_queries   = [query] + generate_subqueries(query)
    all_lists: list[list[dict]] = []

    for q in all_queries:
        for ns in namespaces:
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            dense, sparse = retrieve_from_namespace(
                index, q, dense_encoder, bm25_encoder, ns, top_k, filt
            )
            all_lists.extend([dense, sparse])

    return reciprocal_rank_fusion(all_lists, k=RRF_K)


print("Multi-query functions defined.")

Multi-query functions defined.


### Full retrieval pipeline

Composes all steps: NLP filter extraction → namespace routing → multi-query + hybrid + RRF
→ cross-encoder rerank → MMR diversity → lost-in-middle reorder.


In [ ]:
from __future__ import annotations
from typing import Any


def retrieve_and_rerank(
    query: str,
    index,
    dense_encoder: SentenceTransformer,
    bm25_encoder: BM25Encoder,
    ce_tokenizer: Any,
    ce_model: Any,
) -> list[dict]:
    """
    Full RAG retrieval pipeline for a free-text query.

    1. NLP filter extraction (town, flat_type, year)
    2. Namespace routing
    3. Multi-query hybrid retrieval + weighted RRF
    4. Cross-encoder rerank (top-50 → top-10)
    5. MMR diversity (top-10 → top-5)
    6. Lost-in-middle reorder

    Returns:
        top-5 reordered context chunks.
    """
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)

    fused = multi_query_retrieve(
        query=query, index=index,
        dense_encoder=dense_encoder, bm25_encoder=bm25_encoder,
        namespaces=namespaces, top_k=TOP_K_RETRIEVAL,
        metadata_filter=metadata_filter,
    )
    reranked = rerank_cross_encoder(
        query=query, candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer, model=ce_model, top_k=TOP_K_RERANK,
    )
    diverse = mmr_filter(
        candidates=reranked, dense_encoder=dense_encoder,
        query=query, top_k=TOP_K_MMR, lambda_param=MMR_LAMBDA,
    )
    return reorder_for_context_window(diverse)[:TOP_K_FINAL]


# ── smoke test ─────────────────────────────────────────────────────────────────
smoke_ctx = retrieve_and_rerank(
    "Is $580k fair for a 4-room flat in Tampines?",
    index, dense_encoder, bm25_encoder, ce_tokenizer, ce_model,
)
print(f"Smoke test: {len(smoke_ctx)} chunks retrieved")
for i, c in enumerate(smoke_ctx, 1):
    m = c.get("metadata") or {}
    print(f"  [{i}] {m.get('source')} | {m.get('town')} | {m.get('flat_type','')} | {m.get('sale_year','')}")

Smoke test: 5 chunks retrieved
  [1] transaction | TAMPINES | 4 ROOM | 2025
  [2] transaction | TAMPINES | 4 ROOM | 2023
  [3] transaction | TAMPINES | 4 ROOM | 2022
  [4] transaction | TAMPINES | 4 ROOM | 2018
  [5] transaction | TAMPINES | 4 ROOM | 2022


: 

### Prediction tool — HybridClusterEnsemble via joblib (v3 fix)

Three changes from v2:
- `MODEL_BUNDLE_PATH` now points at `hybrid_cluster_bundle.joblib`
- `load_model_bundle()` loads the object directly — no dict check
- `run_prediction()` calls `.predict()` on the bundle object itself

Gemma 3 extracts `PredictRequest`-shaped fields from the free-text query.
Requires at minimum `town` + `flat_type` to attempt a prediction.


In [ ]:
from __future__ import annotations
import json, re
import joblib
import pandas as pd
import ollama


def load_model_bundle(path: str) -> object | None:
    """
    Load the HybridClusterEnsemble from hybrid_cluster_bundle.joblib.

    Returns the bundle object directly (not a dict).
    Returns None if file not found.
    """
    if not os.path.exists(path):
        print(f"  Model bundle not found: {path}")
        return None
    bundle = joblib.load(path)
    print(f"  Model bundle loaded: {type(bundle).__name__}")
    return bundle


def extract_predict_request(
    query: str,
    model: str = OLLAMA_MODEL,
) -> dict | None:
    """
    Use Gemma 3 to extract a PredictRequest-shaped dict from a free-text query.

    Target fields (matching PropertyLens PredictRequest schema):
        town, flat_type, floor_area_sqm, storey_range,
        lease_commence_date, flat_model

    Returns None if town + flat_type cannot be extracted.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract fields for a Singapore HDB price prediction.
Return ONLY a JSON object with these keys (omit if not mentioned):
  town               : ALL CAPS HDB town e.g. "TAMPINES"
  flat_type          : e.g. "4 ROOM", "3 ROOM"
  floor_area_sqm     : number (convert sqft to sqm if needed: sqft × 0.0929)
  storey_range       : string e.g. "07 TO 09"
  lease_commence_date: integer year lease started e.g. 1990
  flat_model         : e.g. "New Generation", "Improved", "Model A"
Return {{}} if nothing clear. Return ONLY JSON.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if (parsed.get("town") and parsed.get("flat_type")) else None
    except Exception as e:
        print(f"  PredictRequest extraction failed: {e}")
        return None


def run_prediction(predict_request: dict, model_bundle: object) -> str:
    """
    Call HybridClusterEnsemble.predict() and return a formatted result string.

    Calls .predict() directly on the bundle object (v3 fix).

    Args:
        predict_request: dict of HDB flat features.
        model_bundle: loaded joblib object.

    Returns:
        Formatted prediction string or error message.
    """
    try:
        row        = pd.DataFrame([predict_request])
        prediction = model_bundle.predict(row)
        price      = float(prediction[0]) if hasattr(prediction, "__len__") else float(prediction)
        low, high  = price * 0.90, price * 1.10
        return (
            f"Model price estimate: SGD {int(round(price)):,} "
            f"(confidence band: SGD {int(round(low)):,} – SGD {int(round(high)):,}). "
            f"Input used: {predict_request}"
        )
    except Exception as e:
        return f"[Prediction error: {type(e).__name__}: {e}]"


def prediction_tool(query: str, model_bundle: object | None) -> str:
    """
    End-to-end prediction: free-text query → price estimate string.

    Returns empty string if model bundle unavailable or fields cannot be extracted.
    """
    if model_bundle is None:
        return ""
    predict_request = extract_predict_request(query)
    if predict_request is None:
        return "[Prediction tool: insufficient fields extracted — skipping.]"
    return run_prediction(predict_request, model_bundle)


# ── load bundle ────────────────────────────────────────────────────────────────
model_bundle = load_model_bundle(MODEL_BUNDLE_PATH)

# ── smoke test ─────────────────────────────────────────────────────────────────
test_pred = prediction_tool(
    "Is $430k fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon, 64 sqm, lease 1978?",
    model_bundle,
)
print("Prediction smoke test:")
print(test_pred)

### Prompt builder + Gemma 3 answer generation (v3 fix)

System prompt now includes rule 6: **only emit a Fair/Above/Below verdict for price
fairness queries** — not for amenity, trend, or XAI explanation queries.


In [ ]:
from __future__ import annotations
from typing import Any
import ollama


def build_system_prompt() -> str:
    """Return the system prompt for Gemma 3 (v3 — verdict rule added)."""
    return """You are a Singapore HDB property pricing assistant for PropertyLens.
Help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context and model prediction (if given). No outside knowledge.
2. Cite every specific claim with [Context N] labels.
3. If a model prediction is provided, reference it explicitly in your answer.
4. If evidence is thin or contradictory, say so clearly.
5. Keep answers to 3-5 sentences unless detail is requested.
6. Give a Fair / Above market / Below market verdict ONLY for price fairness questions.
   For amenity, trend, or explanation questions, do NOT give a price verdict.
"""


def build_rag_prompt(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
) -> str:
    """
    Build the user-turn prompt with labelled context chunks and optional prediction.

    Args:
        query: original user question.
        context_chunks: retrieved + reranked chunks.
        prediction_result: output of prediction_tool() — empty string to omit.

    Returns:
        Formatted prompt string.
    """
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md  = c.get("metadata") or {}
        txt = str(md.get("parent_text") or "").strip()
        hdr = f"[Context {i}] source={md.get('source')} town={md.get('town')} year={md.get('sale_year')}"
        parts.extend([hdr, txt, ""])

    if prediction_result:
        parts.extend(["## Model prediction", prediction_result, ""])

    parts.extend(["## Question", query])
    return "\n".join(parts).strip()


def generate_answer(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
    model: str = OLLAMA_MODEL,
) -> str:
    """
    Generate a grounded answer using Ollama + Gemma 3.

    Args:
        query: original user question.
        context_chunks: final context from retrieve_and_rerank().
        prediction_result: from prediction_tool().

    Returns:
        Answer string.
    """
    _set_ollama_host(OLLAMA_BASE_URL)
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user",   "content": build_rag_prompt(query, context_chunks, prediction_result)},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


print("Prompt builder and generate_answer defined.")

### End-to-end demo — 8 queries across all personas

Covers: Buyer, Seller, Trends, Amenities, XAI, PropertyGuru listing, Cross-source, Comparison.
No metadata passed — everything is automatic.
Print format is source-aware: each chunk type shows its own meaningful fields.


In [ ]:
from __future__ import annotations

DEMO_QUERIES = [
    {
        "query":   "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
        "persona": "Buyer",
    },
    {
        "query":   "What should I list my 5-room Bishan flat for given current market trends?",
        "persona": "Seller",
    },
    {
        "query":   "Are HDB prices in Queenstown rising or falling over the last 3 years?",
        "persona": "Trends",
    },
    {
        "query":   "What amenities are near Bedok North? Any MRT stations or schools?",
        "persona": "Amenities",
    },
    {
        "query":   "Why did the model predict a high price for this Queenstown flat? What features drove it?",
        "persona": "XAI",
    },
    {
        # PropertyGuru listing from the screenshot
        "query":   "Is $430,000 fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon? "
                   "64 sqm, 52 years lease remaining, lease started 1978, "
                   "3 mins walk to Serangoon MRT.",
        "persona": "PropertyGuru listing",
    },
    {
        # Cross-source: needs transactions + trends + amenities together
        "query":   "Should I buy a 4-room flat in Tampines or Bedok? "
                   "Compare prices, trends, and nearby amenities.",
        "persona": "Cross-source comparison",
    },
    {
        # Counterfactual / negotiation
        "query":   "The seller is asking $650k for a 5-room in Sengkang. "
                   "What is a reasonable counter-offer based on recent sales?",
        "persona": "Negotiation",
    },
]


def _print_chunk(i: int, c: dict) -> None:
    """Print a source-aware one-line summary for a context chunk."""
    m      = c.get("metadata") or {}
    source = m.get("source", "")
    if source == "transaction":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] transaction | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")
    elif source == "amenity":
        print(f"    [{i}] amenity | {m.get('town')} | {m.get('amenity_type')} | count={m.get('count')}")
    elif source == "trend":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] trend | {m.get('town')} | median={rp_str} | {m.get('sale_year')}")
    elif source == "xai":
        preview = str(m.get("parent_text", ""))[:80]
        print(f"    [{i}] xai | type={m.get('xai_type')} | {preview}...")
    else:
        print(f"    [{i}] {source} | {m}")


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end and print a formatted result."""
    query = demo["query"]
    print(f"\n{'='*60}")
    print(f"[{demo['persona']}]")
    print(f"{query}")
    print(f"{'='*60}")

    # step 1: show extracted filter + namespaces
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)
    print(f"  Filter     : {metadata_filter}")
    print(f"  Namespaces : {namespaces}")

    # step 2: retrieve + rerank
    ctx = retrieve_and_rerank(
        query=query, index=index,
        dense_encoder=dense_encoder, bm25_encoder=bm25_encoder,
        ce_tokenizer=ce_tokenizer, ce_model=ce_model,
    )
    print(f"\n  Context chunks ({len(ctx)}):")
    for i, c in enumerate(ctx, 1):
        _print_chunk(i, c)

    # step 3: prediction tool
    pred = prediction_tool(query, model_bundle)
    if pred:
        print(f"\n  Prediction : {pred[:140]}{'...' if len(pred) > 140 else ''}")

    # step 4: generate answer
    answer = generate_answer(query, ctx, pred)
    print(f"\n  Answer:\n{answer}")


for demo in DEMO_QUERIES:
    run_demo(demo)

### Known limitations and next steps

#### Known limitations
- School quality not included — amenities contain proximity only, not MOE ranking/oversubscription.
- `extract_predict_request()` requires Gemma 3 to reliably extract `floor_area_sqm` — queries without area fall back to no prediction.
- Buyer retrieval consistency — with `SAMPLE_FOR_TESTING=True` the 1k random sample varies; switch to `False` for full ingest.
- Gemma 3 (local) smaller than GPT-4 class; complex multi-hop reasoning may be weaker.
- Pinecone free tier index size limits — use paid tier for full 1.57M transaction ingest.

#### Suggested next steps
- Set `SAMPLE_FOR_TESTING = False` and run full ingest.
- Expose `retrieve_and_rerank()` + `prediction_tool()` as `/api/rag/query` in `backend/main.py`.
- Add MOE school popularity data (Phase 2A oversubscription) to amenities CSVs.
- Evaluate retrieval quality with a labelled test set using NDCG@5.
- Fine-tune BGE-M3 on PropertyLens domain queries for better retrieval precision.
